<img src="logo.png" alt="Vegeta" width="240">

# Fixed-wing propeller — cruise efficiency, thrust available, engine-out, and what it hands on

The 9×6 two-blade propeller of the twin-motor fixed wing from notebook 09, on its 2212 motor and 3S
battery. The questions a fixed-wing propeller has to answer are different from a quad's: efficiency at
the cruise advance ratio, thrust available against drag over the speed range, climb margin, and whether
**one** motor can hold level flight. Results are drawn and **exported** (`_runs/propeller/fw_9x6.json`
plus CAD files) for the mission, vibration and fatigue notebooks.

Same explicit, coarse assumptions as notebook 11: generic section, axial inflow, rigid blades.

In [ ]:
import json, math, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from vegeta import boreas, dedalus
from vegeta.dedalus import viz as dviz
from vegeta.dedalus.examples import Propeller as PropellerCAD

RUNS = Path("_runs/propeller"); RUNS.mkdir(parents=True, exist_ok=True)
RHO = 1.2

# ---- hand-copied from notebook 09 (mass budget and CFD at 4 deg): keep in sync by hand, on purpose ----
AUW_KG = 1.35                  # all-up weight
WING_AREA = 0.17               # m^2
CL_CFD, CD_CFD, V_CFD = 0.40, 0.070, 14.0     # whole-aircraft RANS point at 4 deg
MOTORS = 2

## 1. Hardware and the aircraft drag polar

The CFD gave one point (Cl, Cd at 4°). A parabolic polar `Cd = Cd0 + k Cl²` is fitted through it with an
assumed Oswald factor, so drag can be evaluated at any speed. That is an assumption to replace with a
CFD angle-of-attack sweep (three more revisions in notebook 09).

In [ ]:
d, p = boreas.inches(9, 6)
prop = boreas.Propeller.from_pitch("9x6 electric", d, p, blades=2, chord_root_m=0.014, chord_max_m=0.022, chord_tip_m=0.006,
                                   mass_kg=0.012, rotor_mass_kg=0.045, notes="generic planform")
airfoil = boreas.Airfoil(name="thin cambered section", cl_alpha=2 * math.pi * 0.9, alpha0_deg=-2.5, cl_max=1.1, cd0=0.02, k=0.04,
                         source="assumed for a 9-inch blade at Re ~ 1.5e5")
motor = boreas.Motor("2212-920KV", kv_rpm_per_volt=920, resistance_ohm=0.12, no_load_current_a=0.6, max_current_a=20, mass_kg=0.055)
battery = boreas.Battery("3S 5000 mAh", cells=3, capacity_ah=5.0, usable_fraction=0.8, mass_kg=0.380)
system = boreas.Propulsion(prop, airfoil, motor, battery, rho=RHO)

AR, OSWALD = 5.9, 0.8
K_INDUCED = 1 / (math.pi * AR * OSWALD)
CD0 = CD_CFD - K_INDUCED * CL_CFD**2
W = AUW_KG * 9.81

def drag(v):                       # level flight: lift = weight -> Cl(v) -> Cd(v) -> D
    q = 0.5 * RHO * v**2
    cl = W / (q * WING_AREA)
    return q * WING_AREA * (CD0 + K_INDUCED * cl**2), cl

print(f"polar: Cd = {CD0:.4f} + {K_INDUCED:.4f} Cl^2  (through the CFD point Cl {CL_CFD}, Cd {CD_CFD})")
print(f"drag at {V_CFD} m/s level flight: {drag(V_CFD)[0]:.2f} N total, {drag(V_CFD)[0] / MOTORS:.2f} N per motor")

## 2. The propeller, drawn

In [ ]:
r = np.array(prop.r); c = np.array(prop.chord); beta = np.array(prop.beta_deg)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].fill_between(r * 1000, -c * 1000 * 0.3, c * 1000 * 0.7, color="#9fb8d0"); ax[0].set_aspect("equal")
ax[0].set(xlabel="radius [mm]", ylabel="chord [mm]", title=f"planform, solidity {prop.solidity:.3f}"); ax[0].grid(alpha=0.3)
ax[1].plot(r * 1000, beta, "o-"); ax[1].set(xlabel="radius [mm]", ylabel="blade angle β [deg]", title=f"twist for {p * 1000:.0f} mm pitch"); ax[1].grid(alpha=0.3)
fig.tight_layout()
cad = PropellerCAD().generate(diameter=d * 1000, pitch=p * 1000, blades=2, hub_diameter=16, hub_height=9, bore=5,
                              chord_root=14, chord_max=22, chord_tip=6, thickness=0.09, camber=0.04)
prop_files = cad.export(RUNS / "fw_9x6_cad", formats=("step", "stl"), stl_tolerance=0.02)
dviz.show(dviz.plot3d(cad))

In [ ]:
fig = dviz.plot_sections(cad, normal="x", positions=[30.0, 65.0, 105.0], cols=3)

## 3. Efficiency against advance ratio — the fixed-wing question

`J = V / (n D)`. The propeller is efficient in a band of J; the cruise point should sit near the peak.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
for rpm in (5000, 7000, 9000):
    Vs = np.linspace(0.5, 22, 30)
    ops = [boreas.solve(prop, airfoil, rpm, v, RHO) for v in Vs]
    J = [o.advance_ratio for o in ops]
    ax[0].plot(J, [o.efficiency for o in ops], label=f"{rpm} rpm")
    ax[1].plot(J, [o.ct for o in ops], label=f"{rpm} rpm")
ax[0].set(xlabel="advance ratio J", ylabel="propeller efficiency", ylim=(0, 1), title="η(J)"); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].set(xlabel="advance ratio J", ylabel="Ct", title="thrust coefficient"); ax[1].legend(); ax[1].grid(alpha=0.3)
fig.tight_layout()

## 4. Thrust available vs drag: speed range, climb margin, engine-out

Thrust available at full throttle (both motors) against the drag curve gives the maximum level speed;
the gap at cruise is the climb margin (`rate of climb ≈ (T − D) V / W`). With one motor out, the
remaining motor's full-throttle thrust must still exceed drag somewhere, or the aircraft cannot hold
altitude.

In [ ]:
Vs = np.linspace(8, 26, 19)
D = np.array([drag(v)[0] for v in Vs])
T_full = np.array([system.at_throttle(1.0, v).thrust for v in Vs])
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(Vs, D, color="#c62828", label="drag (level flight)")
ax[0].plot(Vs, MOTORS * T_full, label="thrust, 2 motors full throttle")
ax[0].plot(Vs, T_full, "--", label="thrust, 1 motor (engine out)")
ax[0].set(xlabel="airspeed [m/s]", ylabel="force [N]", title="thrust available vs required"); ax[0].legend(); ax[0].grid(alpha=0.3)
roc = (MOTORS * T_full - D) * Vs / W
ax[1].plot(Vs, roc); ax[1].axhline(0, color="k", lw=0.8)
ax[1].set(xlabel="airspeed [m/s]", ylabel="rate of climb [m/s]", title="climb at full throttle"); ax[1].grid(alpha=0.3)
fig.tight_layout()
stall_v = math.sqrt(2 * W / (RHO * WING_AREA * 1.1))
v_max = Vs[np.where(MOTORS * T_full > D)[0].max()] if np.any(MOTORS * T_full > D) else float("nan")
eo = np.where(T_full > D)[0]
print(f"stall speed (Cl_max 1.1 assumed): {stall_v:.1f} m/s | max level speed ~{v_max:.0f} m/s | best climb {roc.max():.1f} m/s at {Vs[roc.argmax()]:.0f} m/s")
print("engine out: " + (f"level flight possible between {Vs[eo.min()]:.0f} and {Vs[eo.max()]:.0f} m/s" if len(eo) else "NOT possible on one motor"))

## 5. Cruise, climb and full-power points; endurance and range

In [ ]:
D_cruise = drag(V_CFD)[0]
cruise = system.for_thrust(D_cruise / MOTORS, V_CFD)
climb = system.at_throttle(1.0, 12.0)
static = system.at_throttle(1.0, 0.0)
one_engine = system.at_throttle(1.0, V_CFD)
P_cruise = MOTORS * cruise.electrical_power
endurance_min = battery.usable_wh / P_cruise * 60
range_km = V_CFD * endurance_min * 60 / 1000
pts = {"cruise 14 m/s": cruise, "climb 12 m/s full": climb, "static full": static, "engine-out cruise (1 motor full)": one_engine}
tbl = pd.DataFrame({k: {"throttle": v.throttle, "rpm": v.rpm, "thrust_N": v.thrust, "current_A": v.current, "electrical_W": v.electrical_power,
                        "prop_eff": v.aero.efficiency, "motor_eff": v.motor_efficiency, "J": v.aero.advance_ratio, "current_limited": v.current_limited}
                    for k, v in pts.items()}).round(3)
print(f"cruise: {P_cruise:.0f} W electrical for both motors -> {endurance_min:.0f} min, {range_km:.0f} km still-air range on {battery.usable_wh:.0f} Wh")
print(f"overall propulsive efficiency at cruise: {cruise.aero.efficiency * cruise.motor_efficiency:.2f} (prop x motor)")
tbl

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].plot(cruise.aero.r * 1000, cruise.aero.dT_dr, label="cruise"); ax[0].plot(climb.aero.r * 1000, climb.aero.dT_dr, label="climb")
ax[0].set(xlabel="radius [mm]", ylabel="dT/dr [N/m]", title="thrust loading"); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(cruise.aero.r * 1000, cruise.aero.alpha_deg, label="cruise"); ax[1].plot(climb.aero.r * 1000, climb.aero.alpha_deg, label="climb")
ax[1].set(xlabel="radius [mm]", ylabel="angle of attack [deg]", title="section incidence"); ax[1].legend(); ax[1].grid(alpha=0.3)
fig.tight_layout()

## 6. What the wing and nacelle will feel

1P and 2P (blade-pass) lines against rpm, and the unbalance force on the nacelle. The engine-out case
matters twice: the remaining motor runs at full rpm (highest excitation), and the thrust is asymmetric
(notebook 09's `engine_out` load case).

In [ ]:
rr = np.linspace(3000, 10000, 15)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(rr, rr / 60, label="1P"); ax[0].plot(rr, 2 * rr / 60, label="2P (blade pass)")
for name, pt in (("cruise", cruise), ("climb", climb)):
    ax[0].axvline(pt.rpm, color="#888", ls=":"); ax[0].text(pt.rpm, 20, name, rotation=90, va="bottom")
ax[0].set(xlabel="rpm", ylabel="frequency [Hz]"); ax[0].legend(); ax[0].grid(alpha=0.3)
for g in (6.3, 2.5):
    ax[1].plot(rr, [boreas.unbalance_force(prop.rotor_mass_kg, x, g) for x in rr], label=f"G {g}")
ax[1].set(xlabel="rpm", ylabel="rotating force [N]", title=f"unbalance, rotor {prop.rotor_mass_kg * 1000:.0f} g"); ax[1].legend(); ax[1].grid(alpha=0.3)
fig.tight_layout()
pd.DataFrame({k: boreas.excitations(prop, v.rpm) for k, v in pts.items()}).round(2)

## 7. Export — the hand-off

In [ ]:
grid = boreas.performance_map(prop, airfoil, np.linspace(3000, 10000, 8), np.linspace(0, 24, 7), RHO)
res = boreas.export(RUNS / "fw_9x6.json", prop, airfoil, map=grid, motor=motor, battery=battery,
                    points={"cruise": cruise, "climb": climb, "static": static, "engine_out": one_engine},
                    notes=f"fixed wing from notebook 09; AUW {AUW_KG} kg, {MOTORS} motors; polar Cd0 {CD0:.4f}, k {K_INDUCED:.4f}; "
                          f"cruise endurance {endurance_min:.0f} min, range {range_km:.0f} km")
print(res)
doc = boreas.load(res.artifacts["json"])
print("files:", sorted(f.name for f in RUNS.glob("fw_9x6*")))
pd.DataFrame({k: {"rpm": v["rpm"], "thrust_N": v["aero"]["thrust"], "current_A": v["current"], "prop_eff": v["aero"]["efficiency"],
                  "shaft_hz": v["excitation"]["shaft_hz"], "blade_pass_hz": v["excitation"]["blade_pass_hz"],
                  "unbalance_N": v["excitation"]["unbalance_force_n"]} for k, v in doc["points"].items()}).round(2)

## 8. CFD check of the cruise point — OpenFOAM, rotating reference frame with axial inflow

`aeromant`'s `rotor_mrf` template: the same CAD propeller in a rotating cell zone with the cruise
airspeed as inflow (steady k-ω SST, MRF). The check is coarse (about 100 k cells, minutes on one core);
expect tens of percent against blade element theory and read the sign message. The CAD axis (Z) is
rotated to +x first. `VEGETA_SKIP_OPENFOAM=1` skips the run when executing headlessly.

In [ ]:
import os
from vegeta import aeromant
from vegeta.aeromant import viz as aviz

RUN_CFD = os.environ.get("VEGETA_SKIP_OPENFOAM") != "1"
prop_x = dedalus.Geometry.from_cadquery(cad.shape.rotate((0, 0, 0), (0, 1, 0), 90), name="prop_axis_x")
stl_x = prop_x.export_stl(RUNS / "fw_9x6_cad" / "prop_axis_x.stl", tolerance=0.02)
CFD_PARAMS = dict(rpm=cruise.rpm, airspeed=V_CFD, diameter=d, kinematic_viscosity=1.5e-5, density=RHO, rotation=1, iterations=400,
                  cells_per_diameter=6.0, surface_level=3, near_level=2, rotor_level=2, wake_level=1)
cfd = None
if RUN_CFD:
    case = aeromant.CFDCase("rotor_mrf", stl_x, CFD_PARAMS, workdir=RUNS / "fw_9x6_cfd_cruise", geometry_units="mm",
                            environment=aeromant.OpenFOAMEnvironment.detect())
    print(case.prepare(overwrite=True))
    cfd = case.run(progress=True)
    print(cfd)
else:
    print("CFD skipped (VEGETA_SKIP_OPENFOAM=1): run this cell on a machine with OpenFOAM to get the check")

In [ ]:
if cfd is not None and cfd.ok:
    m = cfd.metrics
    compare = pd.DataFrame({"BEMT (Boreas)": {"thrust_N": cruise.thrust, "torque_Nm": cruise.aero.torque, "power_W": cruise.aero.power, "efficiency": cruise.aero.efficiency},
                            "CFD (rotor_mrf)": {"thrust_N": m["thrust_N"], "torque_Nm": m["torque_Nm"], "power_W": m["power_W"], "efficiency": m["efficiency"]}})
    compare["CFD / BEMT"] = compare["CFD (rotor_mrf)"] / compare["BEMT (Boreas)"]
    print(f"{m['mesh_cells']} cells, {'converged' if m['converged'] else 'not converged'} in {m['iterations']} iterations; J = {m['advance_ratio']:.2f}")
    display(compare.round(3))

In [ ]:
if cfd is not None and cfd.ok:
    aviz.show(aviz.plot_field_slice(case, "U", normal="z"))
    fig = aviz.plot_section(case, "p", normal="z", zoom=2)
    aviz.show(aviz.plot_streamlines(case, n=80, normal_plane="z"))
    aviz.show(aviz.plot_surface_pressure(case))

In [ ]:
if cfd is not None and cfd.ok:
    doc = json.loads((RUNS / "fw_9x6.json").read_text())
    doc["cfd_cruise"] = {"template": "rotor_mrf", "parameters": CFD_PARAMS, "metrics": {k: v for k, v in cfd.metrics.items() if not isinstance(v, (list, dict))}}
    (RUNS / "fw_9x6.json").write_text(json.dumps(doc, indent=2, default=float))
    print("cfd_cruise added to", RUNS / "fw_9x6.json")

**Next:** mission segments (cruise / climb / loiter) take rpm, current and thrust from these points;
the wing vibration check takes the 1P/2P lines and the unbalance force at the nacelle; the fatigue
spectrum takes the engine-out case as its asymmetric block. Replace the fitted polar with a CFD
angle-of-attack sweep and the generic section with a measured one before trusting absolute numbers.